# Hybrid Energy Model - Getting Started

This notebook walks through the whole pipeline:

1. Generate (synthetic) hourly **load**, **solar**, and **wind** profiles for a year
2. Build a PyPSA network: solar + wind + battery + grid connection
3. Optimize component sizes (capacity expansion) to minimize total annual cost
4. Look at the results: optimal sizes, cost breakdown, and hour-by-hour dispatch

Everything below uses **made-up example data** so you can run it right away.
Once you're comfortable, swap in your own CSVs (see the README).

In [6]:
import matplotlib.pyplot as plt

from hybrid_energy import data, model, plotting

%matplotlib inline
plt.rcParams["figure.figsize"] = (11, 4)

ModuleNotFoundError: No module named 'hybrid_energy'

## 1. Generate example profiles (one full year, hourly)

In [ ]:
N_HOURS = 24 * 365  # one year, hourly resolution

load_kw = data.synthetic_load_profile(n_hours=N_HOURS)
solar_cf = data.synthetic_solar_cf(n_hours=N_HOURS)
wind_cf = data.synthetic_wind_cf(n_hours=N_HOURS)

load_kw.head()

In [ ]:
# Sanity-check the shapes over a sample week
window = slice("2024-01-01", "2024-01-08")

fig, axes = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
load_kw.loc[window].plot(ax=axes[0], title="Load (kW)")
solar_cf.loc[window].plot(ax=axes[1], title="Solar capacity factor")
wind_cf.loc[window].plot(ax=axes[2], title="Wind capacity factor")
plt.tight_layout()

## 2. Build the network

One electrical bus with:
- a **solar** generator and a **wind** generator (both size-optimizable)
- a **battery** (`StorageUnit`, size-optimizable, 4-hour duration)
- a **grid connection** you can import from (at a per-MWh price)
- a small "unmet demand" slack generator with a very high cost, just so the
  optimization always has a feasible fallback instead of failing outright

Cost assumptions live in `hybrid_energy/model.py::DEFAULT_COSTS` - they're
illustrative placeholders, not real quotes. Override any of them by passing
a `costs=` dict to `build_network`.

In [ ]:
network = model.build_network(
    load_kw=load_kw,
    solar_cf=solar_cf,
    wind_cf=wind_cf,
    battery_max_hours=4.0,
    grid_available=True,
)

status, condition = model.optimize_capacity(network)
print(status, condition)

## 3. Results: optimal sizing and cost breakdown

In [ ]:
model.capacity_summary(network)

In [ ]:
model.cost_summary(network)

In [ ]:
plotting.plot_capacities(network)
plt.tight_layout()

## 4. Hour-by-hour dispatch (a sample week)

In [ ]:
plotting.plot_dispatch(network, start="2024-07-01", end="2024-07-08")
plt.tight_layout()

In [ ]:
plotting.plot_state_of_charge(network, start="2024-07-01", end="2024-07-08")
plt.tight_layout()

## Next steps

- **Use your own data:** replace the three `data.synthetic_*` calls above with
  `data.load_profile_csv("../data/your_file.csv")` for load, solar, and wind.
- **Change the system:** try `grid_available=False` for an off-grid design, or
  change `battery_max_hours`.
- **Change the economics:** pass your own `costs={...}` dict to `build_network`
  (see `DEFAULT_COSTS` in `hybrid_energy/model.py` for the keys).
- **Compare scenarios:** build two networks with different assumptions and
  compare `model.cost_summary(...)` side by side.